# Document Q&A (RAG) with ChromaDB + Claude

Ask questions about a set of local text documents and get grounded answers back.

Pipeline:
1. Load `.txt` documents from `./news_articles`
2. Chunk them with overlap
3. Embed chunks locally with `sentence-transformers` and store them in a persistent ChromaDB collection
4. Take a question via `input()`, retrieve the most relevant chunks
5. Ask Claude to answer *using only the retrieved chunks* and print the result

Note on the two reference scripts this is built from: the OpenAI version calls the embeddings API once per chunk in a Python loop, which is slow and burns API calls unnecessarily — sentence-transformers batches embedding locally for free, so that problem doesn't carry over. The Claude summarizer's XML-tag-and-parse pattern is reused below for the final answer so failures are visible instead of silently returning `None`.


In [3]:
%pip install -q anthropic python-dotenv chromadb sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import re
from textwrap import dedent

from dotenv import load_dotenv
from anthropic import Anthropic

import chromadb
from chromadb.utils import embedding_functions

In [5]:
load_dotenv()

client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
model = "claude-sonnet-4-6"


def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})


def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

## 1. Load documents

Reads every `.txt` file from `./news_articles`. Three sample documents (solar cars, EV charging
infrastructure, and transformer architecture) are included so the notebook runs end-to-end
out of the box — swap in your own `.txt` files in that folder to use different sources.

In [8]:
def load_documents_from_directory(directory_path):
    print("==== Loading documents from directory ====")
    documents = []
    for filename in sorted(os.listdir(directory_path)):
        if filename.endswith(".txt"):
            with open(os.path.join(directory_path, filename), "r", encoding="utf-8") as file:
                documents.append({"id": filename, "text": file.read()})
    return documents


directory_path = "./news_articles"
documents = load_documents_from_directory(directory_path)
print(f"Loaded {len(documents)} documents: {[d['id'] for d in documents]}")

if len(documents) < 3:
    raise RuntimeError(
        f"Expected at least 3 .txt documents in {directory_path}, found {len(documents)}. "
        "Add more .txt files to that folder."
    )

==== Loading documents from directory ====


FileNotFoundError: [Errno 2] No such file or directory: './news_articles'

## 2. Chunk with overlap

Character-based chunking with overlap so a fact split across a chunk boundary still shows up
intact in at least one chunk. Each chunk keeps a pointer back to its source document id.

In [ ]:
def split_text(text, chunk_size=1000, chunk_overlap=150):
    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap must be smaller than chunk_size")

    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start = end - chunk_overlap
    return chunks


chunked_documents = []
for doc in documents:
    chunks = split_text(doc["text"])
    for i, chunk in enumerate(chunks):
        chunked_documents.append({
            "id": f"{doc['id']}_chunk{i + 1}",
            "source": doc["id"],
            "text": chunk,
        })

print(f"Split {len(documents)} documents into {len(chunked_documents)} chunks")

## 3. Embed and store in ChromaDB

Uses `sentence-transformers` (`all-MiniLM-L6-v2`) as a local embedding model — no external API
calls or per-chunk cost, and Chroma's `SentenceTransformerEmbeddingFunction` batches the whole
collection in one pass instead of looping one chunk at a time. Storage is persistent, so
re-running this cell just re-upserts the same IDs rather than duplicating data.

In [ ]:
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

chroma_client = chromadb.PersistentClient(path="chroma_persistent_storage")
collection = chroma_client.get_or_create_collection(
    name="document_qa_collection", embedding_function=embedding_fn
)

collection.upsert(
    ids=[c["id"] for c in chunked_documents],
    documents=[c["text"] for c in chunked_documents],
    metadatas=[{"source": c["source"]} for c in chunked_documents],
)

print(f"Collection now holds {collection.count()} chunks")

## 4. Retrieve relevant chunks

In [ ]:
def query_documents(question, n_results=3):
    results = collection.query(query_texts=[question], n_results=n_results)
    chunks = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    print("==== Retrieved chunks ====")
    for meta, dist in zip(metadatas, distances):
        print(f"  - {meta['source']} (distance: {dist:.3f})")

    return chunks, metadatas

## 5. Generate a grounded answer with Claude

Same structured-XML pattern as the document summarizer: Claude is told to answer only from the
supplied context, say so explicitly if the answer isn't in it, and respond inside tags so a
regex failure is visible instead of silently returning nothing.

In [ ]:
RAG_SYSTEM_PROMPT = dedent("""
    You are a question-answering assistant. You will be given retrieved context chunks and a
    question. Answer the question using ONLY information from the provided context.

    If the context does not contain enough information to answer the question, say so explicitly
    instead of guessing or using outside knowledge.

    Respond using exactly this format, with no text outside the tags:

    <answer>
    Your answer, in three sentences maximum.
    </answer>

    <grounded>yes|no</grounded>
""").strip()


def build_context(chunks, metadatas):
    parts = []
    for chunk, meta in zip(chunks, metadatas):
        parts.append(f"[Source: {meta['source']}]\n{chunk}")
    return "\n\n".join(parts)


def generate_response(question, chunks, metadatas):
    context = build_context(chunks, metadatas)
    user_message = f"Context:\n{context}\n\nQuestion:\n{question}"

    messages = []
    add_user_message(messages, user_message)
    return chat(messages, system=RAG_SYSTEM_PROMPT, temperature=0)


def parse_answer(raw):
    answer_match = re.search(r"<answer>(.*?)</answer>", raw, re.DOTALL)
    grounded_match = re.search(r"<grounded>(.*?)</grounded>", raw, re.DOTALL)

    answer = answer_match.group(1).strip() if answer_match else raw.strip()
    grounded = grounded_match.group(1).strip().lower() if grounded_match else "unknown"

    return {"answer": answer, "grounded": grounded}

## 6. Ask a question

In [ ]:
question = input("Ask a question about the documents: ")

chunks, metadatas = query_documents(question)
raw_response = generate_response(question, chunks, metadatas)
parsed = parse_answer(raw_response)

print("\nANSWER")
print("-" * 60)
print(parsed["answer"])
print(f"\n(grounded in retrieved context: {parsed['grounded']})")